# Notebook 05 – Candidate Pattern Discovery

Discover **candidate** UI adaptation patterns from the cleaned survey dataset using FP-Growth.

This notebook produces **two** outputs and performs pattern discovery **only** (no scoring, no confidence-based quality filtering, no JSON):

1. **Base candidate profiles** keyed by `Persona + Mood + Device` — the main UI configuration.
2. **Trait modifiers** keyed by Big Five levels — ordinal nudges that refine specific UI properties.

Personality does **not** define a separate interface. Persona, mood, and device drive the base configuration; Big Five traits adjust the *style* of that configuration. Notebook 06 will merge, score, and consolidate; Notebook 07 exports the repositories.

## Approach

**Base profiles**
- Antecedents contain only `Persona`, `Mood`, `Device`; consequents contain only UI variables.
- Rules are mined per UI category (Global, Desktop, Mobile), concatenated, and rules with the same antecedent are merged into one candidate profile (UI adaptations = union of consequent UI items).
- A shared `min_support` is raised incrementally (`0.05 → … → 0.15`) until the profile count is manageable (≤ 500).

**Trait modifiers**
- Each Big Five level nudges configurable properties (information density, whitespace, visual richness, recommendation emphasis, animation level) by `+1` / `-1`.
- **Data-driven** where the survey shows a clear ordinal shift; **theory** fallback otherwise. Provenance is recorded per row.

## Expected Outputs
- `reports/AssociationRules/candidate_base_profiles.csv` / `.xlsx`
- `reports/AssociationRules/trait_modifiers.csv` / `.xlsx`
- `reports/AssociationRules/base_candidate_rules.csv`

In [1]:
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def _bootstrap_project() -> Path:
    search_from = Path.cwd().resolve()
    candidates = [search_from, *search_from.parents]
    nested_root = search_from / "EvidenceBasedAdaptiveUI"
    if (nested_root / "src" / "config.py").exists():
        candidates.insert(0, nested_root)
    for candidate in candidates:
        if (candidate / "src" / "config.py").exists():
            root = str(candidate)
            if root not in sys.path:
                sys.path.insert(0, root)
            return candidate
    raise FileNotFoundError("Could not find project root containing src/config.py.")


_bootstrap_project()

from src.association_rules.pipeline import run_candidate_patterns_pipeline
from src.association_rules.transactions import build_transactions
from src.utils.dependencies import ensure_packages
from src.utils.notebook import setup_notebook

ensure_packages("mlxtend")

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)
plt.rcParams["figure.dpi"] = 300

PATHS, REPORTS = setup_notebook("AssociationRules")

## Load Clean Dataset

In [2]:
df = pd.read_csv(PATHS.data_processed / "clean_dataset.csv")
logger.info("Loaded clean dataset: %s rows", len(df))
display(df.head(3))

INFO: Loaded clean dataset: 200 rows


,timestamp,consent,age_group,gender,education_level,primary_device,shopping_motivation,decision_speed,price_sensitivity,review_importance,...,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness,Extraversion_Level,Agreeableness_Level,Conscientiousness_Level,Neuroticism_Level,Openness_Level
0,20/01/2026 17:24:30,Yes,18-24,Male,Graduate,Smartphone,Research (I like exploring products and learni...,Quick (5-15 minutes - I know what I want),Moderately important (I balance price and qual...,Sometimes read reviews (For certain product ty...,...,3.0,3.0,3.0,3.0,3.0,Medium,Medium,Medium,Medium,Medium
1,20/01/2026 17:27:08,Yes,25-34,Female,Graduate,Smartphone,Deal-hunting (I look for sales and discounts),Moderate (15-30 minutes - I compare a few opti...,Moderately important (I balance price and qual...,Often read reviews (For most purchases),...,3.5,4.5,3.5,2.5,2.5,Medium,High,Medium,Low,Low
2,20/01/2026 17:38:38,Yes,18-24,Male,Graduate,Smartphone,Entertainment (I browse for fun and enjoyment),Impulsive (Less than 5 minutes - I decide quic...,Somewhat unimportant (Price matters but isn't ...,Sometimes read reviews (For certain product ty...,...,4.0,3.0,3.0,3.5,3.0,High,Medium,Medium,Medium,Medium


## Build Transactions

In [3]:
transactions = build_transactions(df)
print(f"Total transactions: {len(transactions)}")
print("Sample transaction:")
for item in transactions[0][:10]:
    print(f"  {item}")

Total transactions: 200
Sample transaction:
  Persona=Impulsive Buyer
  Mood=Neutral
  Device=Smartphone
  Extraversion=Medium
  Agreeableness=Medium
  Conscientiousness=Medium
  Neuroticism=Medium
  Openness=Medium
  Global_font_style_pref=2. Classic Serif (Traditional fonts with deco...
  Global_font_size_pref=2. Medium (14-16px - Standard size, balanced)


## Discover Candidate Patterns

In [4]:
result = run_candidate_patterns_pipeline(df, REPORTS)
base_profiles = result.base_profiles
trait_modifiers = result.trait_modifiers
summary = result.summary

print(f"Chosen min_support: {result.min_support}")
print("Base rules mined per UI category:")
display(result.stage_summary)

INFO: Built 200 full transactions
INFO: Global_UI support=0.05 confidence=0.60 -> 7157 raw rules, 63 context->UI rules, 5098 itemsets
INFO: Desktop_UI support=0.05 confidence=0.60 -> 16762 raw rules, 94 context->UI rules, 8103 itemsets
INFO: Mobile_UI support=0.05 confidence=0.60 -> 146620 raw rules, 334 context->UI rules, 30783 itemsets
INFO: support=0.05 -> 491 base rules, 34 base profiles


Chosen min_support: 0.05
Base rules mined per UI category:


,UI_Category,Min_Support,Min_Confidence,Frequent_Itemsets,Raw_Rules,Base_Rules
0,Global_UI,0.05,0.6,5098,7157,63
1,Desktop_UI,0.05,0.6,8103,16762,94
2,Mobile_UI,0.05,0.6,30783,146620,334


## Summary Statistics

In [5]:
print(f"Total transactions: {summary['total_transactions']}")
print(f"Base candidate rules: {summary['base_rules']}")
print(f"Base candidate profiles (Persona+Mood+Device): {summary['base_profiles']}")
print(
    f"Trait modifiers: {summary['trait_modifiers']} "
    f"({summary['modifiers_data_driven']} data-driven, {summary['modifiers_theory']} theory)"
)
print(f"Average support: {summary['average_support']:.4f}")
print(f"Average confidence: {summary['average_confidence']:.4f}")
print(f"Average lift: {summary['average_lift']:.4f}")

Total transactions: 200
Base candidate rules: 491
Base candidate profiles (Persona+Mood+Device): 34
Trait modifiers: 17 (5 data-driven, 12 theory)
Average support: 0.1052
Average confidence: 0.7213
Average lift: 1.2071


## Base Candidate Profiles

Each profile is one `Persona + Mood + Device` combination with its merged UI adaptations.

In [6]:
profile_columns = [
    "Profile_ID",
    "Persona",
    "Mood",
    "Device",
    "Num_UI_Adaptations",
    "Num_Supporting_Rules",
    "Avg_Support",
    "Avg_Confidence",
    "Avg_Lift",
]
display(base_profiles[profile_columns].head(20))

,Profile_ID,Persona,Mood,Device,Num_UI_Adaptations,Num_Supporting_Rules,Avg_Support,Avg_Confidence,Avg_Lift
0,22,Deal Hunter,,Smartphone,10,34,0.083235,0.756684,1.585946
1,26,Deal Hunter,,,10,34,0.089559,0.746324,1.557234
2,9,,Neutral,Laptop/Desktop,11,24,0.055208,0.736111,1.310721
3,10,Impulsive Buyer,,,12,24,0.096250,0.687500,1.220837
4,8,Impulsive Buyer,,Smartphone,12,24,0.085417,0.683333,1.212351
5,5,Minimalist,Neutral,,15,23,0.061957,0.688406,1.185856
6,6,,Happy,,12,22,0.062727,0.737968,1.294438
7,4,Loyal Customer,,Smartphone,12,21,0.063810,0.708995,1.181011
8,11,,Bored,,15,19,0.060526,0.672515,1.163676
9,25,Researcher,,Laptop/Desktop,8,16,0.055000,0.785714,1.298656


## Example Base Profile

The strongest base profile and its merged UI adaptations.

In [7]:
top_profile = base_profiles.iloc[0]
context = ", ".join(
    f"{label}={top_profile[label]}"
    for label in ["Persona", "Mood", "Device"]
    if isinstance(top_profile[label], str) and top_profile[label].strip()
)
print(f"Context: {context}")
print(f"Supporting rules: {top_profile['Num_Supporting_Rules']}")
print(f"Avg confidence: {top_profile['Avg_Confidence']:.3f} | Avg lift: {top_profile['Avg_Lift']:.3f}")
print("\nMerged UI adaptations:")
for adaptation in top_profile["UI_Adaptations"].split(" | "):
    print(f"  - {adaptation}")

Context: Persona=Deal Hunter, Device=Smartphone
Supporting rules: 34
Avg confidence: 0.757 | Avg lift: 1.586

Merged UI adaptations:
  - Desktop_info_density=Moderate (Balanced - some details visible, cl...
  - Desktop_persistent_filters=Yes (Always visible sidebar with filters)
  - Desktop_whitespace=Balanced
  - Global_whitespace_pref=Balanced (Moderate spacing - comfortable midd...
  - Mobile_image_text_ratio=Balanced (50% images, 50% text - equal emphasis)
  - Mobile_info_density=Moderate (Balanced - some details visible, cl...
  - Mobile_price_display=Strike-through Original (Shows original price...
  - Mobile_sticky_header=Yes (Header stays at top while scrolling)
  - Mobile_touch_size=Large (Easier to tap, less content visible)
  - Mobile_whitespace=Balanced


## Trait Modifiers

Big Five levels nudge configurable UI properties. `Provenance` marks whether each nudge is data-driven or a theory-informed fallback.

In [8]:
display(trait_modifiers)

,Trait,Level,Property,Nudge,Direction,Delta,Group_N,Baseline_Score,Group_Score,Provenance
0,Agreeableness,High,recommendation_emphasis,1,increase,NaN,94,0.7275,0.7287,theory
1,Agreeableness,Low,recommendation_emphasis,1,increase,0.0850,24,0.7275,0.8125,data-driven
2,Conscientiousness,High,information_density,1,increase,NaN,66,0.5350,0.5227,theory
3,Conscientiousness,Low,information_density,-1,decrease,NaN,35,0.5350,0.5500,theory
4,Extraversion,High,animation_level,1,increase,NaN,29,NaN,NaN,theory
5,Extraversion,High,information_density,-1,decrease,-0.0522,29,0.5350,0.4828,data-driven
6,Extraversion,High,recommendation_emphasis,-1,decrease,-0.0723,29,0.7275,0.6552,data-driven
7,Extraversion,Low,recommendation_emphasis,-1,decrease,NaN,61,0.7275,0.7131,theory
8,Neuroticism,High,animation_level,-1,decrease,NaN,45,NaN,NaN,theory
9,Neuroticism,High,information_density,-1,decrease,NaN,45,0.5350,0.5556,theory


## Exports

In [9]:
print("Export locations:")
for name, path in result.export_paths.items():
    print(f"- {name}: {path}")

Export locations:
- candidate_base_profiles_csv: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/AssociationRules/candidate_base_profiles.csv
- candidate_base_profiles_xlsx: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/AssociationRules/candidate_base_profiles.xlsx
- trait_modifiers_csv: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/AssociationRules/trait_modifiers.csv
- trait_modifiers_xlsx: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/AssociationRules/trait_modifiers.xlsx
- base_candidate_rules_csv: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/AssociationRules/base_candidate_rules.csv
- support_histogram: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/AssociationRules/figures/support_histogram.png
- confidence_histogram: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/AssociationRules/figures/confidence_histogram.png
- top_rules_by_lif